In [2]:
import numpy as np
import pandas as pd
import ot
import matplotlib.pyplot as plt
import seaborn as sns
import random
import os
import glob 
from functions import *
from skbio.stats.distance import DistanceMatrix, permanova, mantel, anosim
from scipy.spatial.distance import pdist, squareform

In [3]:
folder = r"C:\Users\Utente\Desktop\progetto vscode\data\data_wol\file_bipatite"
path_files = glob.glob(os.path.join(folder, "*.csv"))

In [5]:
def create_adj_matrix(file):
    # Carico dataframe
    df = pd.read_csv(file)
    df_data = df.iloc[:, 1:]
    
    # tolgo caratteri strani, li cambio con nan e poi con 0
    df_data = df_data.apply(pd.to_numeric, errors='coerce')
    df_data = df_data.fillna(0)
    
    # estraggo la matrice di incidenza: uso direttamente df_data senza iloc
    B = df_data.to_numpy() 
    #B = (B > 0).astype(int)
    n_plants, m_pollinators = B.shape
    
    # creo la matrice di adiacenza
    total_size = n_plants + m_pollinators
    adj_matrix = np.zeros((total_size, total_size))
  
    adj_matrix[:n_plants, n_plants:] = B
    
    adj_matrix[n_plants:, :n_plants] = B.T
    
                
    return adj_matrix

#esempio
adj = create_adj_matrix(r"C:\Users\Utente\Desktop\progetto vscode\data\data_wol\file_bipatite/M_PL_070.csv")
print(adj)

[[ 0.  0.  0.  0.  0.  0.  0.  0. 25. 12. 15. 22.  9.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  4.  5.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.  0.  0.  0.  7.  0.  2.  8.  0. 11.  9.  6.]
 [ 0.  0.  0.  0.  0.  0.  0.  0.  8.  7.  7. 11.  9.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.  0.  0.  0. 13.  8.  8. 16.  9.  5.  6.  9.]
 [ 0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  3.  4.  0.  5.  6.  4.]
 [ 0.  0.  0.  0.  0.  0.  0.  0. 15.  5. 11. 20. 11.  6.  5.  4.]
 [ 0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  3.  5.  0.  0.  0.  0.]
 [25.  0.  7.  8. 13.  0. 15.  0.  0.  0.  0.  0.  0.  0.  0.  0.]
 [12.  0.  0.  7.  8.  0.  5.  0.  0.  0.  0.  0.  0.  0.  0.  0.]
 [15.  4.  2.  7.  8.  3. 11.  3.  0.  0.  0.  0.  0.  0.  0.  0.]
 [22.  5.  8. 11. 16.  4. 20.  5.  0.  0.  0.  0.  0.  0.  0.  0.]
 [ 9.  0.  0.  9.  9.  0. 11.  0.  0.  0.  0.  0.  0.  0.  0.  0.]
 [ 0.  0. 11.  0.  5.  5.  6.  0.  0.  0.  0.  0.  0.  0.  0.  0.]
 [ 0.  0.  9.  0.  6.  6.  5.  0.  0.  0.  0.  0.  0.  0.  0. 

In [6]:
def get_bip_data(file):
    df = pd.read_csv(file)
    
    piante = df.iloc[:, 0].tolist()
    pollinatori = df.columns[1:].tolist()
    
    return piante, pollinatori

In [7]:
metadata = pd.read_csv(r"C:\Users\Utente\Desktop\progetto vscode\data\data_wol\reference/references.csv")

In [8]:
island_mapping = {
    # Continental Island 
    'Amami-Ohsima Island, Japan': 'continental island',
    'Arima Valley': 'continental island',
    "Arthur's Pass, New Zealand": 'continental island',
    'Ashu, Kyoto, Japan': 'continental island',
    'Bristol, England': 'continental island',
    'Cass, New Zealand': 'continental island',
    'Chiloe, Chile': 'continental island',
    'Craigieburn, New Zealand': 'continental island',
    'Hazen Camp, Ellesmere Island, Canada': 'continental island',
    'Hickling, Norfolk, UK': 'continental island',
    'Kibune, Kyoto, Japan': 'continental island',
    'Kyoto City, Japan': 'continental island',
    'Matamata': 'continental island',
    'Melville Island, Canada': 'continental island',
    'Morne Seychellois National Park, Mahé': 'continental island',
    'Mt. Kushigata, Yamanashi Pref., Japan': 'continental island',
    'Mt. Yufu, Japan': 'continental island',
    'Nakaikemi marsh, Fukui Prefecture, Japan': 'continental island',
    'Shelfanger, Norfolk, UK': 'continental island',
    'Tundra, Greenladn': 'continental island',
    'Uummannaq Island, Greenland': 'continental island',
    'Zackenberg': 'continental island',
    
    # Oceanic Island 
    'Black River Gorges National Park, Mauritius': 'oceanic island',
    'Flores, Açores': 'oceanic island',
    'Galapagos': 'oceanic island',
    'Garajonay, Gomera, Spain': 'oceanic island',
    'Mauritius Island': 'oceanic island',
    'Morant Point, Jamaica': 'oceanic island',
    'Puerto Villamil, Isabela Island, Galapagos': 'oceanic island',
    'Syndicate, Dominica': 'oceanic island',
    'Tenerife, Canary Islands': 'oceanic island',
    'Windsor, The Cockpit Country, Jamaica': 'oceanic island'
}

metadata['island_type'] = metadata['Locality of Study'].map(island_mapping).fillna('mainland')

In [16]:
region_mapping = {
    # NEOTROPICAL (South America, Central America, and the Caribbean) 
    'Amarante, Pampas, Argentina': 'Neotropical',
    'Antonio Porto Jatai': 'Neotropical',
    'Arima Valley': 'Neotropical',
    'Atlantic Forest, high elevation': 'Neotropical',
    'Atlantic Forest, low elevation': 'Neotropical',
    'Atlantic Forest, mid elevation': 'Neotropical',
    'Calarca, Quindio': 'Neotropical',
    'Cambucá, Ubatuba': 'Neotropical',
    'Canaima Nat. Park, Venezuela': 'Neotropical',
    'Chiloe, Chile': 'Neotropical',
    'Cinco Cerros, Pampas, Argentina': 'Neotropical',
    'Cordón del Cepo, Chile': 'Neotropical',
    'Diamantina': 'Neotropical',
    'Difuntito, Pampas, Argentina': 'Neotropical',
    'Difuntos, Pampas, Argentina': 'Neotropical',
    'El Morro, Pampas, Argentina': 'Neotropical',
    'El Triunfo 1, Biosphere Reserve': 'Neotropical',
    'El Triunfo 2, Biosphere Reserve': 'Neotropical',
    'Estacion de Biologia Chamela, Jalisco': 'Neotropical',
    'Galapagos': 'Neotropical',
    'Guarico State, Venezuela': 'Neotropical',
    'La Barrosa, Pampas, Argentina': 'Neotropical',
    'La Brava, Pampas, Argentina': 'Neotropical',
    'La Chata, Pampas, Argentina': 'Neotropical',
    'La Paja, Pampas, Argentina': 'Neotropical',
    'Laguna Diamante, Mendoza, Argentina': 'Neotropical',
    'Mindo Lindo': 'Neotropical',
    'Morant Point, Jamaica': 'Neotropical',
    'Nahuel Huapi National Park, Argentina': 'Neotropical',
    'Nanegal 1': 'Neotropical',
    'Nanegal 2': 'Neotropical',
    'Parque Estadual Carlos Botelho': 'Neotropical',
    'Parque Nacional Chiribiquete': 'Neotropical',
    'Parque Nacional do Catimbau': 'Neotropical',
    'Picinguaba, Ubatuba': 'Neotropical',
    'Piedra Alta, Pampas, Argentina': 'Neotropical',
    'Puerto Villamil, Isabela Island, Galapagos': 'Neotropical',
    'Reserva Florestal Mata do Paraíso, Brazil': 'Neotropical',
    'Rio Blanco, Mendoza, Argentina': 'Neotropical',
    'Salento, Quindio': 'Neotropical',
    'Santa Virginia Field Station, Serra do Mar State Park': 'Neotropical',
    'Santuario de Flora y Fauna Galeras': 'Neotropical',
    'Serra da Mantiqueira, PNI, SE Brazil': 'Neotropical',
    'Serra do Cipo National Park': 'Neotropical',
    'Serra do Mar State Park': 'Neotropical',
    'Serra do Mar and Serra da Mantiqueira': 'Neotropical',
    'Serra do Pará, Brazil': 'Neotropical',
    'Syndicate, Dominica': 'Neotropical',
    'Unchog, Carpish Mountains': 'Neotropical',
    'Vigilancia, Pampas, Argentina': 'Neotropical',
    'Volcan, Pampas, Argentina': 'Neotropical',
    'Windsor, The Cockpit Country, Jamaica': 'Neotropical',

    # PALEARCTIC (Europe, North Africa, North and Central Asia)
    'Ashu, Kyoto, Japan': 'Palearctic',
    'Bristol, England': 'Palearctic',
    'Daphní, Athens, Greece': 'Palearctic',
    'Denmark': 'Palearctic',
    'Doñana Nat. Park, Spain': 'Palearctic',
    'Flores, Açores': 'Palearctic',
    'Garajonay, Gomera, Spain': 'Palearctic',
    'Hestehaven, Denmark': 'Palearctic',
    'Hickling, Norfolk, UK': 'Palearctic',
    'Isenbjerg': 'Palearctic',
    'Kibune, Kyoto, Japan': 'Palearctic',
    'Kyoto City, Japan': 'Palearctic',
    'Latnjajaure, Abisko, Sweden': 'Palearctic',
    'Mt. Kushigata, Yamanashi Pref., Japan': 'Palearctic',
    'Mt. Yufu, Japan': 'Palearctic',
    'Nakaikemi marsh, Fukui Prefecture, Japan': 'Palearctic',
    'Parc Natural del Cap de Creus': 'Palearctic',
    'Shelfanger, Norfolk, UK': 'Palearctic',
    'Tenerife, Canary Islands': 'Palearctic',

    # NEARTICA (North America and Greenland)
    'Brownfield, Illinois, USA': 'Nearctic',
    'Carlinville, Illinois, USA': 'Nearctic',
    'Central New Brunswick, Canada': 'Nearctic',
    'Hazen Camp, Ellesmere Island, Canada': 'Nearctic',
    'Highland temperate mosaic forest, Central Mexico': 'Nearctic',
    'Melville Island, Canada': 'Nearctic',
    'Montgomery County, Maryland, USA': 'Nearctic',
    'North Carolina, USA': 'Nearctic',
    'Ottawa, Canada': 'Nearctic',
    'Pikes Peak, Colorado, USA': 'Nearctic',
    'Tundra, Greenladn': 'Nearctic', # Ho mantenuto il typo 'Greenladn' per sicurezza del match
    'Uummannaq Island, Greenland': 'Nearctic',
    'Zackenberg': 'Nearctic',

    # AFROTROPICAL (Sub-Saharan Africa and the West Indies)
    'Black River Gorges National Park, Mauritius': 'Afrotropical',
    'KwaZulu-Natal region, South Africa': 'Afrotropical',
    'Mauritius Island': 'Afrotropical',
    'Morne Seychellois National Park, Mahé': 'Afrotropical',

    #  AUSTRALASIA (Australia, New Zealand)
    "Arthur's Pass, New Zealand": 'Australasian',
    'Cass, New Zealand': 'Australasian',
    'Craigieburn, New Zealand': 'Australasian',
    'Matamata': 'Australasian',
    'Snowy Mountains, Australia': 'Australasian',

    # INDONESIAN (Southeast Asia)
    'Amami-Ohsima Island, Japan': 'Indomalayan'
}

metadata['region'] = metadata['Locality of Study'].map(region_mapping)

Prende in input il dizionario degli attributi dei nodi e restituisce 
    un dizionario in cui le chiavi sono i generi e i valori i nodi associati

In [9]:
def create_gen_partition(node_attributes):
    partition = {}
    
    for i, node_data in node_attributes.items():
        g = node_data["genus"]
        
        if g not in partition:
            partition[g] = []
        partition[g].append(i)
        
    return partition

Creo una lista con tutti i grafi. Ogni grafo è un dizionario rappresentato da: 
- matrice di adiacenza pesata
- attributi: ogni nodo è etichettato con il nome completo, se è pianta o impollinatore, il genere
- la partizione dovuta al genere: un dizionario in cui la chiave è il genere e i valori una lista dei nodi presenti con quel genere
- il tipo di isola
- la regione biogeografica 
- la latitudine

In [57]:
all_g = []

for file in path_files:
    pl, po = get_bip_data(file)
    gg = create_adj_matrix(file)
    
    nodes_names = pl + po 
    node_attributes = {}

    for i, name in enumerate(nodes_names):
        node_type = "Plant" if i < len(pl) else "Pollinator"
        
        # Estraiamo il genere (prima parola)
        g = name.split()[0] 
        
        node_attributes[i] = {
            "name": name,
            "type": node_type,
            "genus": g  
        }
    
    partition_indices = create_gen_partition(node_attributes)
    
    island_types = metadata['island_type'].values
    i = island_types[path_files.index(file)-1]
    region = metadata['region'].values
    r=region[path_files.index(file)-1]
    latitude = metadata['Latitude'].values
    l=latitude[path_files.index(file)-1]
    
    all_g.append({
        "matrix": gg,
        "attributs": node_attributes,
        "genus_part": partition_indices,
        "island_type": i,
        'region': r,
        'latitude':l
    })


Confronta due grafi e crea una partizione basata sui generi comuni.
Se un genere è presente in entrambi, mantiene il nome del genere.
Altrimenti, raggruppa i nodi in classi macro ('Macro_Plant' o 'Macro_Impollinator').

In [ ]:
def create_joint_partitions(g1, g2):
    
    attr_key = "attributs"
    
    attr1 = g1[attr_key]
    attr2 = g2[attr_key]

    gen_g1 = {data["genus"] for data in attr1.values()}
    gen_g2 = {data["genus"] for data in attr2.values()}
    gen_in_both = gen_g1.intersection(gen_g2)

    def make_new_partition(attributi):
        partition = {}
        for node_id, data in attributi.items():
            g = data["genus"]
            node_type = data["type"] 
            
            if g in gen_in_both:
                classe = g
            # se il genere non è in entrambe creo una macro classe 
            else:
                classe = f"Macro_{node_type}"

            if classe not in partition:
                partition[classe] = []
            
            partition[classe].append(node_id)
            
        return partition

    new_part_g1 = make_new_partition(attr1)
    new_part_g2 = make_new_partition(attr2)

    return new_part_g1, new_part_g2

In [12]:
def node_emb(g, partition, dim_embedding, seed=42):
    rdn_state = np.random.get_state()
    np.random.seed(seed)
    
    embedding = NodeEmbedding(g, dim=dim_embedding, k=1, verbose=False)
    X_total = embedding.X
    
    # DIFESA 1: Trasforma eventuali valori NaN o Inf in zeri
    X_total = np.nan_to_num(X_total, nan=0.0, posinf=0.0, neginf=0.0)
    
    X_list = [X_total[list(nodes), :] for nodes in partition]
    np.random.set_state(rdn_state)
    return X_list

In [13]:
def w2_distance(X_list,Y_list):
    # Validate the number of partitions
    m = len(X_list)
    if len(Y_list) != m:
        raise ValueError("The two graphs must have the same number of classes.")
    D = []

    # For cycles for the upper triangular part and diagonal blocks
    for i in range(m):
        for j in range(i, m):
            # Compute the dot product between class i and class j
            A_block = X_list[i] @ X_list[j].T
            B_block = Y_list[i] @ Y_list[j].T

            if i == j:
                if len(A_block) > 1 and len(B_block) > 1:
                    # If we are on a diagonal block, we take only the upper triangular part, excluding the main diagonal fixing k=1
                    upper_tri_indices_A = np.triu_indices_from(A_block, k=1)
                    upper_tri_indices_B= np.triu_indices_from(B_block, k=1)
                    vals_A = A_block[upper_tri_indices_A]
                    vals_B = B_block[upper_tri_indices_B]
                else:
                    continue
            else:
                # For blocks outside the diagonal we take all the elements
                vals_A = A_block.flatten()
                vals_B = B_block.flatten()

            # Calculate the 1D Wasserstein distance between the two distributions
            wd2 = ot.wasserstein_1d(vals_A, vals_B, p=2)**(1/2)
            D.append(wd2)
            
    # Convert the list of block distances into a NumPy array
    D_vector = np.array(D)

    # Calculate the final global distance as the Euclidean norm of the block distances vector
    finale_distance = np.linalg.norm(D_vector)

    return finale_distance

In [14]:
def list_part(g1,g2):
    part_g1=[]
    part_g2=[]
    part1,part2= create_joint_partitions(g1,g2)
    for k1, v1 in part1.items():
        part_g1.append(v1)
    for k2, v2 in part2.items():
        part_g2.append(v2)
    return part_g1 , part_g2
    

In [62]:
a,b=list_part(all_g[17],all_g[36])
print(len(a))
print(len(b))

32
31


In [61]:
p1,p2=create_joint_partitions(all_g[17],all_g[36])
print(p2)
for i in p1.keys():
    if i not in p2.keys():
        print(i)

{'Allium': [0], 'Calystegia': [1], 'Cirsium': [2], 'Eupatorium': [3], 'Juncus': [4], 'Lychnis': [5], 'Silene': [6], 'Sonchus': [7, 8], 'Stachys': [9], 'Macro_Pollinator': [10, 17], 'Bombus': [11, 12, 13, 14], 'Ceuthorrhynchus': [15], 'Chrysomela': [16], 'Eoseristalis': [18], 'Episyrphus': [19], 'Epuraea': [20], 'Eumerus': [21], 'Leptura': [22], 'Melanostoma': [23], 'Meligethes': [24], 'Mesapamea': [25], 'Nemophora': [26], 'Olibrus': [27], 'Pieris': [28], 'Psithyrus': [29], 'Scaeva': [30], 'Strangalia': [31], 'Syrphus': [32, 33], 'Thymelicus': [34], 'Unidentified': [35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48], 'Zygaena': [49]}
Macro_Plant


In [31]:
N = 40
distance_matrix = np.zeros((N, N))
dim_embedding = 12

for i in range(N):
    for j in range(i + 1, N):
        g1 = all_g[i]
        g2 = all_g[j]
        
        part1, part2 = list_part(g1, g2)
        
        # qualche partizione 
        if len(part1) != len(part2):
            continue 
            
        emb1 = node_emb(g1["matrix"], part1, dim_embedding)
        emb2 = node_emb(g2["matrix"], part2, dim_embedding)
        
        d = w2_distance(emb1, emb2)
        distance_matrix[i, j] = d
        distance_matrix[j, i] = d


PERMANOVA: island type (Oceanic, Continental, Mainland)

In [60]:

N = 40
island_types_subset = [all_g[i]['island_type'] for i in range(N)]
sample= [str(i) for i in range(N)]
skbio_distance_matrix = DistanceMatrix(distance_matrix, sample)

permanova_res = permanova(skbio_distance_matrix, island_types_subset, permutations=999)

print(permanova_res)

method name               PERMANOVA
test statistic name        pseudo-F
sample size                      40
number of groups                  3
test statistic            -1.084861
p-value                       0.781
number of permutations          999
Name: PERMANOVA results, dtype: object
